In [ ]:
%pip install openai langchain langchain-community chromadb sentence-transformers pypdf

In [ ]:
import os
import json
import time
import re
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TRIỆU HỒI TOÀN BỘ SỨC MẠNH CỦA SAGEMATH VÀO KERNEL PYTHON 
from sage.all import *

# ==========================================
# 1. CẤU HÌNH API LLAMA 3.1 70B (ĐIỀN LẠI KEY CỦA BẠN VÀO ĐÂY)
# ==========================================
API_KEY = "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".strip()

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = API_KEY
)

PDF_PATH = "/Users/bhsoft/Downloads/projects/ula-workbook.pdf"
SO_CAU_MOI_DANG = 20

# ==========================================
# 2. KHỞI TẠO RAG
# ==========================================
def setup_rag_database(pdf_path):
    print("📚 Đang nạp giáo trình ULA PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return Chroma.from_documents(documents=splits, embedding=embeddings).as_retriever(search_kwargs={"k": 2})

try:
    rag_retriever
    print("✅ RAG: Sách đã được nạp từ trước.")
except NameError:
    rag_retriever = setup_rag_database(PDF_PATH)
    print("✅ RAG: Nạp sách thành công!")

# ==========================================
# 3. BẢN ĐỒ SƯ PHẠM MCQ
# ==========================================
DANH_SACH_MCQ = [
    {"id": "1.2_1.3", "topic": "Khử Gauss và RREF", "action": "rref_calc", "trap": "Tạo 3 đáp án sai do: Quên chia pivot về 1, sai dấu khi cộng trừ hàng, dừng ở dạng REF thay vì rút gọn hẳn lên trên (RREF)."},
    {"id": "2.1_2.2", "topic": "Nhân Ma trận - Vectơ", "action": "mat_vec_mul", "trap": "Tạo 3 đáp án sai do: Nhân sai hàng với cột, cộng nhầm vị trí, hoặc báo lỗi không nhân được do sai kích thước."},
    {"id": "2.5_2.6", "topic": "Biến đổi Hình học", "action": "transform_geom", "trap": "Tạo 3 đáp án sai do: Đảo lộn dấu âm dương trong ma trận (ví dụ nhầm vị trí dấu trừ của hàm sin khi xoay 90 độ)."},
    {"id": "3.1", "topic": "Tìm Ma trận nghịch đảo", "action": "inverse_calc", "trap": "Tạo 3 đáp án sai do: Quên nhân với 1/det(A), hoặc quên đổi dấu phần bù đại số (không chuyển vị)."},
    {"id": "3.2", "topic": "Đổi Cơ sở", "action": "change_basis", "trap": "Tạo 3 đáp án sai do: Nhân ngược ma trận chuyển cơ sở (nhân với P thay vì P^-1 hoặc ngược lại)."},
    {"id": "3.4", "topic": "Định thức chứa tham số", "action": "det_param", "trap": "Tạo 3 đáp án sai do: Sai dấu khi khai triển phần bù đại số, hoặc bình phương sai tham số m."},
    {"id": "4.1_4.2", "topic": "Giá trị riêng", "action": "eigen_calc", "trap": "Tạo 3 đáp án sai do: Giải sai phương trình đặc trưng (sai dấu lambda), hoặc tính nhầm đường chéo chính."},
    {"id": "4.5", "topic": "Chuỗi Markov", "action": "markov_calc", "trap": "Tạo 3 đáp án sai: Một đáp án có tổng các phần tử không bằng 1, một đáp án quên không nhân ma trận 2 lần."},
    {"id": "5.2", "topic": "Phương pháp lũy thừa", "action": "power_method", "trap": "Tạo 3 đáp án sai do: Quên chuẩn hóa (normalize) vectơ sau vòng lặp, hoặc tính sai tích vô hướng."},
    {"id": "6.1_6.3", "topic": "Hình chiếu trực giao", "action": "projection", "trap": "Tạo 3 đáp án sai do: Dùng sai công thức hình chiếu (chia cho độ dài của sai vectơ nền, thay vì vectơ chiếu lên)."},
    {"id": "6.4", "topic": "Trực giao hóa Gram-Schmidt", "action": "gram_schmidt", "trap": "Tạo 3 đáp án sai do: Cộng thay vì trừ thành phần hình chiếu, hoặc chia sai độ dài chuẩn."},
    {"id": "7.1_7.2", "topic": "Dạng toàn phương", "action": "quad_form", "trap": "Tạo 3 đáp án sai do: Chọn nhầm hướng ứng với giá trị riêng nhỏ nhất (thay vì lớn nhất để tối đa hóa)."},
    {"id": "7.4", "topic": "Phân tích Giá trị suy biến (SVD)", "action": "svd_calc", "trap": "Tạo 3 đáp án sai do: Lấy luôn trị riêng của A^T*A làm đáp án mà quên không lấy căn bậc hai (sqrt) để ra sigma."}
]

# ==========================================
# 4. LÕI TOÁN HỌC BẰNG SAGEMATH
# ==========================================
def generate_math_data(action):
    try:
        if action == "rref_calc":
            A = random_matrix(ZZ, 3, 4, x=-2, y=4)
            rref_mat = A.rref()
            return f"Cho ma trận mở rộng A = {latex(A)}. Tính dạng bậc thang rút gọn (RREF) của A.", f"RREF(A) = {latex(rref_mat)}"
        
        elif action == "mat_vec_mul":
            A = random_matrix(ZZ, 2, 2, x=-3, y=4)
            x = vector([2, -1])
            return f"Tính tích Ax với ma trận A = {latex(A)} và vectơ x = {latex(x)}.", f"Ax = {latex(A*x)}"
        
        elif action == "transform_geom":
            theta = pi / 2
            rot = matrix([[cos(theta), -sin(theta)], [sin(theta), cos(theta)]])
            return "Tìm ma trận chuẩn A biểu diễn phép biến đổi tuyến tính: Quay một vectơ góc 90 độ ngược chiều kim đồng hồ quanh gốc tọa độ.", f"A = {latex(rot)}"
        
        elif action == "inverse_calc":
            A = random_matrix(QQ, 2, 2)
            while A.det() == 0: A = random_matrix(QQ, 2, 2)
            return f"Tìm ma trận nghịch đảo A^{{-1}} của A = {latex(A)}.", f"A^{{-1}} = {latex(A.inverse())}"
        
        elif action == "change_basis":
            P = matrix([[1, -1], [1, 1]]) 
            x_B = vector([2, 3])
            return f"Cho cơ sở B. Ma trận chuyển cơ sở từ B sang cơ sở chuẩn là P_B = {latex(P)}. Tìm tọa độ của vectơ x trong cơ sở chuẩn, biết [x]_B = {latex(x_B)}.", f"x = P_B * [x]_B = {latex(P*x_B)}"
        
        elif action == "det_param":
            var('m')
            A = matrix([[m, 2, 0], [1, -1, 1], [0, 3, m]])
            return f"Cho ma trận A = {latex(A)}. Tính định thức det(A) theo tham số m.", f"det(A) = {latex(A.det())}"
        
        elif action == "eigen_calc":
            A = matrix([[2, 1], [0, 3]])
            return f"Tìm các giá trị riêng (eigenvalues) của ma trận A = {latex(A)}.", f"Các giá trị riêng là \\lambda_1 = 2, \\lambda_2 = 3"
        
        elif action == "markov_calc":
            P = matrix(QQ, [[8/10, 3/10], [2/10, 7/10]])
            x0 = vector([1, 0])
            return f"Cho ma trận xác suất chuyển trạng thái Markov P = {latex(P)} và trạng thái ban đầu x_0 = {latex(x0)}. Tính trạng thái x_2 sau 2 bước lặp.", f"x_2 = P^2 * x_0 = {latex(P^2 * x0)}"
        
        elif action == "power_method":
            A = matrix([[4, 1], [1, 3]])
            x0 = vector([1, 0])
            return f"Áp dụng Phương pháp lũy thừa cho ma trận A = {latex(A)} với vectơ khởi tạo x_0 = {latex(x0)}. Tính x_1 (chưa chuẩn hóa).", f"x_1 = A * x_0 = {latex(A * x0)}"
        
        elif action == "projection":
            u = vector([3, -1, 2])
            v = vector([1, 1, 1])
            proj = (u.inner_product(v) / v.inner_product(v)) * v
            return f"Tìm hình chiếu trực giao của vectơ u = {latex(u)} lên phương của vectơ v = {latex(v)}.", f"Hình chiếu = {latex(proj)}"
        
        elif action == "gram_schmidt":
            v1 = vector([1, 1, 0])
            v2 = vector([1, 0, 1])
            u1 = v1
            u2 = v2 - (v2.inner_product(u1) / u1.inner_product(u1)) * u1
            return f"Áp dụng quá trình Gram-Schmidt cho hệ cơ sở v_1 = {latex(v1)}, v_2 = {latex(v2)}. Tìm vectơ trực giao u_2 (biết u_1 = v_1).", f"u_2 = {latex(u2)}"
        
        elif action == "quad_form":
            A = matrix([[5, 0], [0, 2]])
            return f"Cho dạng toàn phương xác định bởi ma trận đối xứng A = {latex(A)}. Hướng nào của vectơ x (khi ||x||=1) sẽ tối đa hóa giá trị của x^T A x?", "Hướng của vectơ riêng tương ứng với trị riêng lớn nhất (\\lambda = 5), tức là vectơ [1, 0]^T."
        
        elif action == "svd_calc":
            A = matrix([[1, 2], [0, 2]])
            AtA = A.transpose() * A
            evals = AtA.eigenvalues()
            sigma1 = sqrt(max(evals))
            sigma2 = sqrt(min(evals))
            return f"Tìm các giá trị suy biến (singular values) của ma trận A, biết A^T A = {latex(AtA)}.", f"Các giá trị suy biến là căn bậc hai của các trị riêng: \\sigma_1 = {latex(sigma1)}, \\sigma_2 = {latex(sigma2)}"
        
        else:
            return "LỖI", "LỖI"
            
    except Exception as e:
        print(f"Lỗi SageMath ({action}): {e}")
        return "LỖI TẠO ĐỀ", "LỖI TẠO ĐÁP ÁN"

# ==========================================
# 5. GỌI API LLAMA 3.1 70B
# ==========================================
def generate_mcq_question(topic, question_math, correct_math, trap_instruction):
    relevant_docs = rag_retriever.invoke(topic)
    context_text = "\n".join([doc.page_content for doc in relevant_docs])
    
    prompt = f"""Bạn là giáo sư Toán học giảng dạy theo sách Understanding Linear Algebra. 
    Lý thuyết nền tảng: {context_text}
    
    Bài toán: {question_math}
    Kết quả chuẩn xác: {correct_math}
    
    Tạo 1 câu trắc nghiệm 4 đáp án (MCQ) bằng TIẾNG VIỆT.
    ⚠️ LUẬT TẠO BẪY ĐÁP ÁN SAI (TUYỆT ĐỐI TUÂN THỦ):
    Không sinh số ngẫu nhiên bừa bãi. Bạn phải tạo 3 đáp án sai dựa CHÍNH XÁC vào các lỗi kinh điển sau: "{trap_instruction}"
    
    ⚠️ LƯU Ý VỀ ĐỊNH DẠNG JSON (QUAN TRỌNG):
    - TUYỆT ĐỐI KHÔNG sử dụng ký tự xuống dòng (enter) bên trong các chuỗi giá trị của JSON.
    - Nếu cần xuống dòng, hãy dùng đúng ký tự `\\n`.
    - Dữ liệu trả về không được chứa bất kỳ ký tự điều khiển ẩn nào.

    Trả về ĐÚNG định dạng JSON sau:
    {{
        "question": "Nội dung câu hỏi",
        "options": ["A. ...", "B. ...", "C. ...", "D. ..."],
        "correct_answer": "Đáp án đúng (A, B, C hoặc D)",
        "explanation": "Giải thích chi tiết tại sao đáp án đó đúng, và phân tích cặn kẽ TỪNG đáp án sai vì sao lại mắc bẫy.",
        "type": "MCQ"
    }}"""
    
    # Sử dụng mô hình Llama 3.1 70B của Meta
    response = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct", 
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,    # Nhiệt độ thấp để cấu trúc JSON chuẩn xác # Ngăn chặn AI ngắt JSON giữa chừng
    )
    return response.choices[0].message.content

# ==========================================
# 6. VÒNG LẶP SẢN XUẤT CHÍNH
# ==========================================
def main():
    print("="*70)
    print("🚀 BẮT ĐẦU SẢN XUẤT MCQ (BẰNG SAGEMATH + LLAMA 3.1 70B)")
    print("="*70)

    kq_mcq = []
    
    for bai in DANH_SACH_MCQ:
        print(f"\n⚙️ Đang xử lý: {bai['topic']}...")
        for i in range(SO_CAU_MOI_DANG):
            try:
                q_math, c_math = generate_math_data(bai["action"])
                if q_math == "LỖI TẠO ĐỀ": continue

                json_res = generate_mcq_question(bai["topic"], q_math, c_math, bai["trap"])
                
                # BƯỚC 1: Lột vỏ markdown
                clean_json = json_res.strip().strip("```").replace("json\n", "").replace("JSON\n", "")
                
                # BƯỚC 2: Diệt ký tự điều khiển (Control characters)
                # Quét và xóa sạch các ký tự ẩn có mã ASCII từ 0-31 (như tab, newline, chuông báo...)
                clean_json = re.sub(r'[\x00-\x1f\x7f]', '', clean_json)
                
                # BƯỚC 3: Màng lọc chống lỗi gạch chéo ngược LaTeX
                clean_json = clean_json.replace('\\', '\\\\').replace('\\\\"', '\\"').replace('\\\\n', '\\n')
                
                # BƯỚC 4: Dịch chuỗi thành Dictionary
                cau_hoi = json.loads(clean_json)

                cau_hoi["id"] = f"MCQ_ULA_{bai['id']}_{i+1:03d}"
                cau_hoi["topic"] = bai["topic"]
                kq_mcq.append(cau_hoi)
                
                print(f"   ✅ Xong câu {i+1}/{SO_CAU_MOI_DANG}")
                time.sleep(2)
            except Exception as e: 
                print(f"   ❌ Lỗi câu {i+1}: {e}")
                # In dòng gây lỗi ra để dễ debug nếu vẫn còn sót (tuỳ chọn)
                # print(f"       JSON lỗi: {clean_json[:100]}...")

    if kq_mcq:
        with open("ngan_hang_MCQ_full_Sage.json", "w", encoding="utf-8") as f:
            json.dump(kq_mcq, f, ensure_ascii=False, indent=4)
        print("\n🎉 HOÀN TẤT! Đã lưu toàn bộ câu hỏi MCQ vào file: ngan_hang_MCQ_full_Sage.json")

if __name__ == "__main__":
    main()